# **In Silico Identification of Novel Compounds for Insomnia disease.-  Step 1A: Retrival of  Human Orexin 2 receptor Antagonists from ChEMBL and ADME peopierties molecule filtering**


##**Author: Maurizio Rafael Hernández Díaz.**

The workflow begins at the retrival of a  bioacivity dataset for the orexin receptor 2  from the CHEMBL database, starting from a specific UniProt ID identified `O43614` while the PDB entry `4S0V` provide the structural basis for the molecular docking and pharmacohphore model. A last step for the chemBL obtained dataset will evaluate molecules based of ADME(Absorption, Distribution, Metabolisim and Excretion) as this rules desrcibe important molecular propierties for a drug drug's pharmacokinetics in the human body.

The ChemBL obtained  dataset will be used as Training  and validation data to develop a  QSAR model.

##**0.-LIBRARIES REQUIRED:**

In [1]:

!pip install rdkit


In [2]:
#Importing required modules from the libraries:
import numpy as np
import matplotlib.pyplot as plt
import rdkit.Chem as chem
from typing import List
from io import StringIO
import pandas as pd
from rdkit.Chem import Descriptors as chemdesc
from rdkit.Chem import PandasTools
from rdkit.Chem import AllChem as achem
PandasTools.InstallPandasTools()
import json
import requests
import pandas as pd
import rdkit.Chem as chem
from rdkit.Chem import AllChem, Descriptors, Lipinski, MACCSkeys
from rdkit import Chem

Failed to patch pandas - PandasTools will have limited functionality


##**1.-ChEMBL ID RETRIVAL**

For this first part we start a UniProt search in order to find the **Histamine 1 receptor(H1)** enzyme in UniProt, after we have located its UniProt ID we are going to make use of the ChEMBL API to try to find the ChEMBL ID making use of requests to the ChEMBL API and obtaining relevant information about the response.

In [3]:
uniprot_id="O43614" #We store the id in a variable we are going to use afterwards

In [4]:
base_url="https://www.ebi.ac.uk/chembl/api/data/{:s}"  #Defining the URL we are going to use to retrive information from ChEMBL.

In order to search un ChEMBL as we start from a Uniprot ID , searching the Documentation we find **search** as mentioned: "Special type of filter allowing a full text search based on elastic search queries"

In [5]:
molecule_url = base_url.format(f"target/search.json?q={uniprot_id}") #We use this search and specify the uniprot ID we defined and storage it in "molecule_url"
molecule_url #In order to see such URL


'https://www.ebi.ac.uk/chembl/api/data/target/search.json?q=O43614'

**After finding it , we must retrive such json in order to find the corresponding ChEMBL ID**

In [6]:
#We found the JSON but now we are going to retrive it from ChEMBL
response = requests.get(molecule_url, headers={"Accept": "application/json"})
# Check if the response is successful; as mentioned we must obtain the 200 status code
assert(response.status_code == 200)
response.status_code # Let´s check again for the 200 status code
molecule_request = response.json()
#molecule_request  if you want to check such JSON file, the information is storaged as similar to a python dictionary, being a nested dictionary.


In [7]:
response = requests.get(molecule_url) #Doing such request
response
assert(response.status_code==200)
response.status_code #If the status code is 200 meaning the request has ben ssuccessful
#We obtain a 200 response status code so we continue :)

200

**Now we must transform said JSON(JavaScript Object Notation) into a Pandas DataFrame for easier management.**

In [8]:
data = response.json()
targets = data.get("targets", [])
df_targets = pd.DataFrame(targets) #
df_targets #tThe final dataframe we are obtaining the ChEMBL ID from "target_chembl_id"

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,Orexin receptor type 2,21.0,False,CHEMBL4792,"[{'accession': 'O43614', 'component_descriptio...",SINGLE PROTEIN,9606
1,[],Homo sapiens,Orexin receptor,19.0,False,CHEMBL3307226,"[{'accession': 'O43614', 'component_descriptio...",PROTEIN FAMILY,9606


In [9]:
chembl_id="CHEMBL4792" #As we did with the Uniprot ID we storage it in a variable.

##**2.- RETRVING ANTAGONIST(INHIBITORY) MOLECULES FOR Human Orexin 2 Receptor**

In this section we will use the previously obtained  **ChEMBL_id** to access the activity data from the active compounds, retrive information for  the compounds IC50 value(half-maximal inhibitory concentration) as this parameter represents the concentration of a drug or compound required to inhibit a specific biological process by 50%, we need to specify that we are tartgeting a H1 histamine receptor in Human(*Homo sapiens*) as well as other precise metrics.

In [10]:

def get_activities(chembl_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/activity?target_chembl_id={chembl_id}&format=json"
    params = {
        "limit": 1000,
        "standard_type": "IC50"
    }

    all_rows = []
    current_url = url

    while current_url:
        response = requests.get(current_url, params=params)
        response.raise_for_status()
        data = response.json()

        all_rows.extend(data["activities"])

        next_rel = data.get("page_meta", {}).get("next")
        if next_rel:
            current_url = f"https://www.ebi.ac.uk{next_rel}"
            params = None
        else:
            break

    df = pd.DataFrame.from_records(all_rows)


    df = df.dropna(subset=["standard_value", "canonical_smiles", "standard_units"])
    df = df[df["standard_units"] == "nM"]
    df["standard_value"] = df["standard_value"].astype(float)
    df = df[df["standard_value"] > 0]
    df["pIC50"] = -np.log10(df["standard_value"] * 1e-9)

    return df[[
        "molecule_chembl_id",
        "canonical_smiles",
        "standard_value",
        "standard_units",
        "pIC50",

    ]].drop_duplicates(subset="molecule_chembl_id")




I will be working with pIC50 values as it aligns with biological systems’ logarithmic response, and simplifies potency comparisons across compounds, aiding in clearer data interpretation.

In [11]:

df_activities = get_activities(chembl_id)
df= df_activities
df

,molecule_chembl_id,canonical_smiles,standard_value,standard_units,pIC50
0,CHEMBL138101,COc1cc2c(cc1OC)CN(C(=O)C(Cc1ccc(Br)cc1)C(C)(C)...,2000.0,nM,5.698970
1,CHEMBL343551,COc1cc2c(cc1OC)CN(C(=O)C(Cc1ccccc1)NC(=O)c1cc(...,36.0,nM,7.443697
2,CHEMBL140504,COc1cc2c(cc1OC)CN(C(=O)CCc1ccccc1)CC2,10000.0,nM,5.000000
3,CHEMBL142009,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1ccncc1)C(C)(...,40.0,nM,7.397940
4,CHEMBL343786,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1ccsc1)C(C)(C...,35.0,nM,7.455932
...,...,...,...,...,...
6491,CHEMBL6160702,COc1ccnc(N2CCC3(CCN(Cc4cn([11CH3])c5ccccc45)C(...,1406.0,nM,5.852015
6492,CHEMBL6166134,Cc1ccc(-n2nccn2)c(C(=O)N2CCCN(c3nc4cc(Cl)ccc4o...,2.0,nM,8.698970
6493,CHEMBL6152751,CC[C@H](C)[C@H]1CN(c2nc3cc(Cl)ccc3o2)CCCN1C(=O...,54.0,nM,7.267606
6495,CHEMBL6195025,CCN(C(=O)c1cccc(F)c1-c1ncccn1)[C@@H](C)CNc1ncc...,140.0,nM,6.853872


In [12]:
def clean_pic50_table(df):
    """
    I created this function in order to filter only the PIC50 values
    """  # We leave only the IC50
    df["pIC50"] = pd.to_numeric(df["pIC50"], errors="coerce")
    df = df[df["pIC50"] > 0]  # Delete 0 values or negatives
    return df


In [13]:
clean_pic50_table(df)
df= df.drop(columns=["standard_value","standard_units"])

In [14]:
Filtered_dataset= df.copy()
Filtered_dataset

,molecule_chembl_id,canonical_smiles,pIC50
0,CHEMBL138101,COc1cc2c(cc1OC)CN(C(=O)C(Cc1ccc(Br)cc1)C(C)(C)...,5.698970
1,CHEMBL343551,COc1cc2c(cc1OC)CN(C(=O)C(Cc1ccccc1)NC(=O)c1cc(...,7.443697
2,CHEMBL140504,COc1cc2c(cc1OC)CN(C(=O)CCc1ccccc1)CC2,5.000000
3,CHEMBL142009,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1ccncc1)C(C)(...,7.397940
4,CHEMBL343786,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1ccsc1)C(C)(C...,7.455932
...,...,...,...
6491,CHEMBL6160702,COc1ccnc(N2CCC3(CCN(Cc4cn([11CH3])c5ccccc45)C(...,5.852015
6492,CHEMBL6166134,Cc1ccc(-n2nccn2)c(C(=O)N2CCCN(c3nc4cc(Cl)ccc4o...,8.698970
6493,CHEMBL6152751,CC[C@H](C)[C@H]1CN(c2nc3cc(Cl)ccc3o2)CCCN1C(=O...,7.267606
6495,CHEMBL6195025,CCN(C(=O)c1cccc(F)c1-c1ncccn1)[C@@H](C)CNc1ncc...,6.853872


##**3.-MOLECULAR FILTERING: ADME PROPERTIES(Lipinski rules)**

As the Lipinksi rules states:  

– The molecule should have no more than 5 H-bond donors and 10 H-bond acceptors (i.e. N or O atoms).

– The molecular weight should be under 500 Daltons.

– Its logP should be 5 maximum.

In [16]:
Filtered_dataset["Mol"] = Filtered_dataset["canonical_smiles"].apply(Chem.MolFromSmiles)
Filtered_dataset
Filtered_dataset= Filtered_dataset[Filtered_dataset["pIC50"]>= 6.5]

In [17]:
def lipinski_filter(mol):
    if mol is None:
        return False


    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)

    conditions = [
        mw < 500,
        logp < 5.0,
        hbd < 5,
        hba < 10
    ]

    return all(conditions)


Filtered_dataset["is_lipinski"] = Filtered_dataset["Mol"].apply(lipinski_filter)

print(Filtered_dataset["is_lipinski"].value_counts())


is_lipinski
True     2596
False    1078
Name: count, dtype: int64


In [18]:
FinaL_dataset=Filtered_dataset[Filtered_dataset["is_lipinski"]==True]

1. ¿Cómo flexibilizar los rangos?
Aquí tienes una propuesta de rangos más realistas para antagonistas de orexina (basándome en los perfiles de Suvorexant y Lemborexant):
MW (Peso): Súbelo de 400 a 450 o incluso 500. Las orexinas tienen muchos anillos y necesitan masa para llenar el bolsillo del receptor.
LogP: Amplíalo a 1.5 - 5.0. El rango 2.0-4.0 es muy estrecho para compuestos que deben ser lipofílicos para cruzar la BHE.
HBD (Donadores): Súbelo a max 2. Permitir un donador más te abre la puerta a muchas amidas secundarias esenciales.
Rotatable Bonds: Súbelo a 10. 7 es demasiado rígido para la librería de Enamine.

Fármaco (Nombre Genérico)	Marca Comercial	Año Aprobación	Peso Molecular (MW)
Suvorexant	Belsomra	2014	450.9 g/mol
Lemborexant	Dayvigo	2019	410.4 g/mol
Daridorexant	Quviviq	2022	450.9 g/mo

In [19]:
df=FinaL_dataset.copy()
df.rename(columns={"Mol":"mol"},inplace=True)


In [20]:
def pajouhesh_lenz_filter(df):
    """
    Filter based on  Pajouhesh & Lenz (2005)
    for succesful CNS drugs.
    """
    df["RotatableBonds"] = df["mol"].apply(Descriptors.NumRotatableBonds)
    df["mw"] = df["mol"].apply(Descriptors.MolWt)
    df["logp"] = df["mol"].apply(Descriptors.MolLogP)
    df["hbd"] =df["mol"].apply(Descriptors.NumHDonors)
    df["hba"] = df["mol"].apply(Descriptors.NumHAcceptors)
    df["TPSA"] = df["mol"].apply(Descriptors.TPSA)
    cns_candidates = df[
        (df['mw'] >= 350) & (df['mw'] <= 500) &
        (df['logp'] >= 1.0) & (df['logp'] <= 5.0) &
        (df['TPSA'] <= 90) &
        (df['hbd'] <= 2) &
        (df['hba'] <= 8) &
        (df['RotatableBonds'] <= 10)
    ].copy()
    return cns_candidates

Orexin_2 = pajouhesh_lenz_filter(df)

In [21]:
from rdkit.Chem import QED
Orexin_2["QED"] = Orexin_2["mol"].apply(QED.qed)
Orexin_2 = Orexin_2[Orexin_2["QED"] > 0.6].copy()

**As we are searching for a pontential drug candiate we use a QED >0.6 which ensures optimal balance of pysicochemical propierties such as molecular weight, lipophilicity, and polar surface area wich are really similar as the molecular profiles of successful oral drugs**

In [22]:
Orexin_2

,molecule_chembl_id,canonical_smiles,pIC50,mol,is_lipinski,RotatableBonds,mw,logp,hbd,hba,TPSA,QED
3,CHEMBL142009,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1ccncc1)C(C)(...,7.397940,<rdkit.Chem.rdchem.Mol object at 0x1767b23b0>,True,6,397.519,3.18800,1,5,63.69,0.811083
4,CHEMBL343786,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1ccsc1)C(C)(C...,7.455932,<rdkit.Chem.rdchem.Mol object at 0x1767b2420>,True,6,402.560,3.85450,1,5,50.80,0.797299
5,CHEMBL139634,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1cccnc1)C(C)(...,6.619789,<rdkit.Chem.rdchem.Mol object at 0x1767b2490>,True,6,397.519,3.18800,1,5,63.69,0.811083
7,CHEMBL140547,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1cccn1C)C(C)(...,7.552842,<rdkit.Chem.rdchem.Mol object at 0x1767b2570>,True,6,399.535,3.13150,1,4,55.73,0.810895
13,CHEMBL344622,COc1cc2c(cc1OC)CN(C(=O)[C@@H](NCc1cccs1)C(C)(C...,7.602060,<rdkit.Chem.rdchem.Mol object at 0x1767b2650>,True,6,402.560,3.85450,1,5,50.80,0.797299
...,...,...,...,...,...,...,...,...,...,...,...,...
6288,CHEMBL5905339,COC(=O)N1[C@H](C)C[C@H](NS(=O)(=O)N(C)C)[C@@H]...,8.080922,<rdkit.Chem.rdchem.Mol object at 0x1776b6f80>,True,7,471.595,2.86230,1,5,88.18,0.660762
6292,CHEMBL5760626,COC(=O)N1[C@H](C)C[C@H](NS(=O)(=O)N(C)C)[C@@H]...,6.966576,<rdkit.Chem.rdchem.Mol object at 0x1776b6ff0>,True,7,489.585,3.00140,1,5,88.18,0.636305
6293,CHEMBL5998404,COC(=O)N1[C@H](C)C[C@H](NS(C)(=O)=O)[C@@H]1CO[...,7.593460,<rdkit.Chem.rdchem.Mol object at 0x1776b7060>,True,6,460.543,3.15460,1,5,84.94,0.705067
6296,CHEMBL5752182,COC(=O)N1[C@H](C)C[C@H](NS(=O)(=O)N(C)C)[C@@H]...,6.728158,<rdkit.Chem.rdchem.Mol object at 0x1776b71b0>,True,7,483.606,2.64630,1,5,88.18,0.644304


In [23]:
Orexin_2.to_csv("../DATA/OX2_receptor_chembl_curated_database_definitive.csv")

In this first step in the workflow  we retrieve compounds that target the Orexin 2 receptor and constructed a database of inhibitory compounds that do match Lipinski rules as favorable molecules that present desireable physiochemical profiles for drug bioavailability, as mentioned previously this Dataset will be used in future steps.